In [24]:
import torch
import numpy as np
from transformers import T5ForConditionalGeneration, AutoTokenizer, T5Config
from datasets import load_dataset
import copy
import time
import json
from tqdm import tqdm
import torch.nn as nn

In [2]:
MODEL_NAME = "Kyrmasch/t5-kazakh-qa"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"Loading model: {MODEL_NAME}")

Device: cpu
Loading model: Kyrmasch/t5-kazakh-qa


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
base_model.eval()
base_model.to(DEVICE)

print(f"Model loaded. Parameters: {sum(p.numel() for p in base_model.parameters()):,}")

tokenizer_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/582k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/home/jovyan/.local/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


config.json:   0%|          | 0.00/812 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

Model loaded. Parameters: 756,155,392


In [13]:
sample_ffn = base_model.encoder.block[0].layer[1].DenseReluDense
if hasattr(sample_ffn, 'wi'):
    D_FF = sample_ffn.wi.weight.shape[0]
elif hasattr(sample_ffn, 'wi_0'):
    D_FF = sample_ffn.wi_0.weight.shape[0]
print(f"FFN hidden size (d_ff): {D_FF}")
print(f"Encoder layers: {len(base_model.encoder.block)}")

FFN hidden size (d_ff): 2816
Encoder layers: 24


In [14]:
CALIBRATION_QA = [
    ("Қазақстанның астанасы қай қала?", "Қазақстан Республикасының астанасы — Астана қаласы. Бұл қала 1997 жылы елордаға айналды."),
    ("Қазақстан қашан тәуелсіздік алды?", "Қазақстан Республикасы 1991 жылы 16 желтоқсанда тәуелсіздігін жариялады."),
    ("Алматы қай жерде орналасқан?", "Алматы — Қазақстанның оңтүстік-шығысындағы ең ірі қаласы. Тяньшань тауларының етегінде жатыр."),
    ("Байқоңыр ғарыш айлағы қашан салынды?", "Байқоңыр ғарыш айлағы 1955 жылы салынды. Ол әлемдегі ең ірі ғарыш айлағы."),
    ("Балқаш көлі қай облыста?", "Балқаш көлі Алматы және Қарағанды облыстарында орналасқан. Үлкен тұщы су көлі."),
    ("Қазақстанның ресми тілі қандай?", "Қазақстанда мемлекеттік тіл — қазақ тілі. Орыс тілі ресми мәртебеге ие."),
    ("Ұлы Жібек жолы не болды?", "Ұлы Жібек жолы — ежелгі сауда жолы. Қытайдан Жерорта теңізіне дейін созылды."),
    ("Қарағанды нені өндіреді?", "Қарағанды — Қазақстанның өнеркәсіп орталығы. Мұнда көмір өндіріледі."),
    ("Семей ядролық полигоны қашан жабылды?", "Семей ядролық полигоны 1991 жылы жабылды. Ол КСРО кезінде жұмыс істеді."),
    ("Атырау қаласы қай өзеннің жанында?", "Атырау қаласы Жайық өзенінің бойында орналасқан. Мұнай өндірісінің орталығы."),
    ("Нұр-Сұлтан қаласы бұрын қалай аталды?", "Нұр-Сұлтан — бұрын Астана деп аталған қала. 2019 жылы атауы өзгертілді."),
    ("Қазақстанда неше облыс бар?", "Қазақстан 17 облысқа бөлінеді. Соның ішінде 3 республикалық маңызы бар қала бар."),
    ("Алтай тауы қай елде орналасқан?", "Алтай тауы Қазақстан, Ресей, Қытай және Моңғолия шекарасында орналасқан."),
    ("Каспий теңізі қандай теңіз?", "Каспий теңізі — дүниедегі ең үлкен жабық су айдыны. Тұзды көл болып саналады."),
    ("Қазақстанның жері қанша?", "Қазақстанның аумағы 2,7 миллион шаршы километрді құрайды. Дүние жүзінде 9-орын алады."),
    ("Тараз қаласының тарихы қандай?", "Тараз — Қазақстандағы ең ежелгі қалалардың бірі. 2000 жылдан астам тарихы бар."),
    ("Шымкент қай өңірде орналасқан?", "Шымкент — Оңтүстік Қазақстанның ірі қаласы. Республикалық маңызы бар қала мәртебесіне ие."),
    ("Іле өзені қайда ағады?", "Іле өзені Қытайдан басталып, Қазақстан арқылы Балқаш көліне құяды."),
    ("Қазақстанның алғашқы президенті кім?", "Қазақстанның алғашқы президенті — Нұрсұлтан Назарбаев. 1991 жылдан 2019 жылға дейін басқарды."),
    ("Астана халықаралық қаржы орталығы не?", "Астана халықаралық қаржы орталығы — AIFC. 2018 жылы ашылған қаржы хабы."),
]

In [15]:
def get_calibration_texts(qa_pairs):
    texts = []
    for q, c in qa_pairs:
        texts.append(f"question: {q} context: {c}")
    return texts

CAL_TEXTS = get_calibration_texts(CALIBRATION_QA)

In [16]:
def collect_ffn_activations(model, texts):
    activations = {}
    hooks = []

    for i, block in enumerate(model.encoder.block):
        ffn = block.layer[1].DenseReluDense

        def make_hook(layer_idx):
            def hook_fn(module, input, output):
                # ReLU активации после wi
                acts = output.detach().cpu().float()
                acts = acts.view(-1, acts.shape[-1])
                if layer_idx not in activations:
                    activations[layer_idx] = []
                activations[layer_idx].append(acts)
            return hook_fn

        if hasattr(ffn, 'wi'):
            hooks.append(ffn.wi.register_forward_hook(make_hook(i)))
        elif hasattr(ffn, 'wi_0'):
            hooks.append(ffn.wi_0.register_forward_hook(make_hook(i)))

    with torch.no_grad():
        for text in tqdm(texts, desc="Collecting activations", leave=False):
            enc = tokenizer(text, return_tensors="pt", max_length=256,
                           truncation=True, padding=True).to(DEVICE)
            dec_ids = torch.zeros((enc['input_ids'].shape[0], 1), dtype=torch.long, device=DEVICE)
            model(**enc, decoder_input_ids=dec_ids)

    for h in hooks:
        h.remove()

    result = {}
    for layer_idx, acts_list in activations.items():
        result[layer_idx] = torch.cat(acts_list, dim=0)

    return result


print("\n--- Collecting FFN activations ---")
ffn_activations = collect_ffn_activations(base_model, CAL_TEXTS)
for li, acts in ffn_activations.items():
    print(f"  Layer {li}: {acts.shape}")


--- Collecting FFN activations ---


  Layer 0: torch.Size([786, 2816])
  Layer 1: torch.Size([786, 2816])
  Layer 2: torch.Size([786, 2816])
  Layer 3: torch.Size([786, 2816])
  Layer 4: torch.Size([786, 2816])
  Layer 5: torch.Size([786, 2816])
  Layer 6: torch.Size([786, 2816])
  Layer 7: torch.Size([786, 2816])
  Layer 8: torch.Size([786, 2816])
  Layer 9: torch.Size([786, 2816])
  Layer 10: torch.Size([786, 2816])
  Layer 11: torch.Size([786, 2816])
  Layer 12: torch.Size([786, 2816])
  Layer 13: torch.Size([786, 2816])
  Layer 14: torch.Size([786, 2816])
  Layer 15: torch.Size([786, 2816])
  Layer 16: torch.Size([786, 2816])
  Layer 17: torch.Size([786, 2816])
  Layer 18: torch.Size([786, 2816])
  Layer 19: torch.Size([786, 2816])
  Layer 20: torch.Size([786, 2816])
  Layer 21: torch.Size([786, 2816])
  Layer 22: torch.Size([786, 2816])
  Layer 23: torch.Size([786, 2816])


In [17]:
def compute_mi_matrix(activations: torch.Tensor) -> torch.Tensor:
    """
    Gaussian MI: MI(i,j) = -0.5 * log(1 - rho_ij^2)
    activations: [N, d_ff]
    """
    A = activations.float()
    A = A - A.mean(0, keepdim=True)
    std = A.std(0, keepdim=True).clamp(min=1e-8)
    A_norm = A / std

    N = A_norm.shape[0]
    corr = (A_norm.T @ A_norm) / N  # [d_ff, d_ff]
    corr = corr.clamp(-1 + 1e-6, 1 - 1e-6)

    mi = -0.5 * torch.log(1 - corr ** 2)
    mi.fill_diagonal_(0.0)
    return mi


def importance_score(activations: torch.Tensor) -> torch.Tensor:
    """
    Важность нейрона = дисперсия активаций (чем выше — тем больше информации несёт).
    activations: [N, d_ff] -> [d_ff]
    """
    return activations.float().var(dim=0)


def greedy_mi_pruning(mi_matrix: torch.Tensor,
                      neuron_importance: torch.Tensor,
                      prune_ratio: float) -> list:
    """
    Правильный жадный алгоритм:
    1. Берём пару (i, j) с максимальным MI[i,j]
    2. Удаляем менее важный нейрон из пары
    3. Исключаем удалённый из дальнейшего рассмотрения
    4. Повторяем до нужного количества

    Возвращает список индексов для удаления.
    """
    d_ff = mi_matrix.shape[0]
    n_prune = int(d_ff * prune_ratio)

    if n_prune == 0:
        return []

    # Копируем MI для модификации
    mi = mi_matrix.clone()
    alive = set(range(d_ff))
    pruned = []

    while len(pruned) < n_prune:
        if len(alive) < 2:
            break

        # Ищем максимум MI среди живых нейронов
        # Маскируем удалённые
        mask = torch.zeros(d_ff, dtype=torch.bool)
        for idx in alive:
            mask[idx] = True

        mi_masked = mi.clone()
        mi_masked[~mask] = -1
        mi_masked[:, ~mask] = -1
        mi_masked.fill_diagonal_(-1)

        # Пара с максимальным MI
        flat_idx = mi_masked.argmax().item()
        i, j = flat_idx // d_ff, flat_idx % d_ff

        if mi_masked[i, j] <= 0:
            # Нет значимого MI — удаляем по важности
            alive_list = sorted(alive)
            imp = neuron_importance[alive_list]
            worst = alive_list[imp.argmin().item()]
            pruned.append(worst)
            alive.remove(worst)
        else:
            # Удаляем менее важный из пары
            if neuron_importance[i] >= neuron_importance[j]:
                to_remove = j
            else:
                to_remove = i

            pruned.append(to_remove)
            alive.remove(to_remove)
            # Обнуляем MI удалённого
            mi[to_remove, :] = -1
            mi[:, to_remove] = -1

    return sorted(pruned)

In [18]:
def build_pruned_model(original_model, pruned_indices_per_layer: dict):
    """
    Физически изменяем размер FFN: создаём новые nn.Linear меньшего размера
    и копируем оставшиеся веса. Это даёт реальное ускорение.
    """
    model = copy.deepcopy(original_model)

    for layer_idx, pruned_indices in pruned_indices_per_layer.items():
        if not pruned_indices:
            continue

        block = model.encoder.block[layer_idx]
        ffn = block.layer[1].DenseReluDense

        pruned_set = set(pruned_indices)
        if hasattr(ffn, 'wi'):
            old_d_ff = ffn.wi.weight.shape[0]
        elif hasattr(ffn, 'wi_0'):
            old_d_ff = ffn.wi_0.weight.shape[0]
        else:
            continue

        keep_indices = [i for i in range(old_d_ff) if i not in pruned_set]
        new_d_ff = len(keep_indices)
        keep_t = torch.tensor(keep_indices, dtype=torch.long)

        with torch.no_grad():
            if hasattr(ffn, 'wi') and hasattr(ffn, 'wo'):
                # T5DenseActDense: wi [d_ff, d_model], wo [d_model, d_ff]
                d_model = ffn.wi.weight.shape[1]

                new_wi = nn.Linear(d_model, new_d_ff, bias=(ffn.wi.bias is not None))
                new_wi.weight.data = ffn.wi.weight.data[keep_t]
                if ffn.wi.bias is not None:
                    new_wi.bias.data = ffn.wi.bias.data[keep_t]

                new_wo = nn.Linear(new_d_ff, d_model, bias=(ffn.wo.bias is not None))
                new_wo.weight.data = ffn.wo.weight.data[:, keep_t]
                if ffn.wo.bias is not None:
                    new_wo.bias.data = ffn.wo.bias.data  # bias не зависит от d_ff

                ffn.wi = new_wi
                ffn.wo = new_wo

            elif hasattr(ffn, 'wi_0') and hasattr(ffn, 'wi_1') and hasattr(ffn, 'wo'):
                # T5DenseGatedActDense: wi_0, wi_1 [d_ff, d_model], wo [d_model, d_ff]
                d_model = ffn.wi_0.weight.shape[1]

                for attr in ['wi_0', 'wi_1']:
                    old_w = getattr(ffn, attr)
                    new_w = nn.Linear(d_model, new_d_ff, bias=(old_w.bias is not None))
                    new_w.weight.data = old_w.weight.data[keep_t]
                    if old_w.bias is not None:
                        new_w.bias.data = old_w.bias.data[keep_t]
                    setattr(ffn, attr, new_w)

                new_wo = nn.Linear(new_d_ff, d_model, bias=(ffn.wo.bias is not None))
                new_wo.weight.data = ffn.wo.weight.data[:, keep_t]
                if ffn.wo.bias is not None:
                    new_wo.bias.data = ffn.wo.bias.data
                ffn.wo = new_wo

    return model


def count_params(model):
    return sum(p.numel() for p in model.parameters())

In [19]:
TEST_QA = [
    {
        "question": "Қазақстанның астанасы қай қала?",
        "context": "Қазақстан Республикасының астанасы — Астана қаласы. Бұл қала 1997 жылы елордаға айналды.",
        "keywords": ["астана", "astana"]
    },
    {
        "question": "Қазақстан қашан тәуелсіздік алды?",
        "context": "Қазақстан Республикасы 1991 жылы 16 желтоқсанда тәуелсіздігін жариялады.",
        "keywords": ["1991"]
    },
    {
        "question": "Алматы қай жерде орналасқан?",
        "context": "Алматы — Қазақстанның оңтүстік-шығысындағы ең ірі қаласы. Тяньшань тауларының етегінде жатыр.",
        "keywords": ["оңтүстік", "шығыс", "тяньшань"]
    },
    {
        "question": "Байқоңыр ғарыш айлағы қашан салынды?",
        "context": "Байқоңыр ғарыш айлағы 1955 жылы салынды. Ол әлемдегі ең ірі ғарыш айлағы.",
        "keywords": ["1955"]
    },
    {
        "question": "Балқаш көлі қай облыста орналасқан?",
        "context": "Балқаш көлі Алматы және Қарағанды облыстарында орналасқан. Үлкен тұщы су көлі.",
        "keywords": ["алматы", "қарағанды"]
    },
    {
        "question": "Қазақстанның ресми тілі қандай?",
        "context": "Қазақстанда мемлекеттік тіл — қазақ тілі. Орыс тілі ресми мәртебеге ие.",
        "keywords": ["қазақ", "kazakh"]
    },
    {
        "question": "Атырау қаласы қай өзеннің жанында?",
        "context": "Атырау қаласы Жайық өзенінің бойында орналасқан. Мұнай өндірісінің орталығы.",
        "keywords": ["жайық", "jayyq", "ural"]
    },
    {
        "question": "Қазақстанның алғашқы президенті кім?",
        "context": "Қазақстанның алғашқы президенті — Нұрсұлтан Назарбаев. 1991 жылдан 2019 жылға дейін басқарды.",
        "keywords": ["назарбаев", "nursultan", "нұрсұлтан"]
    },
    {
        "question": "Қарағанды нені өндіреді?",
        "context": "Қарағанды — Қазақстанның өнеркәсіп орталығы. Мұнда көмір өндіріледі.",
        "keywords": ["көмір", "coal"]
    },
    {
        "question": "Семей ядролық полигоны қашан жабылды?",
        "context": "Семей ядролық полигоны 1991 жылы жабылды. Ол КСРО кезінде жұмыс істеді.",
        "keywords": ["1991"]
    },
]

In [20]:
def evaluate_model(model, test_data, n_runs=3):
    model.eval()
    correct = 0
    total = len(test_data)
    times = []

    with torch.no_grad():
        for item in test_data:
            input_text = f"question: {item['question']} context: {item['context']}"
            inputs = tokenizer(input_text, return_tensors="pt",
                             max_length=512, truncation=True).to(DEVICE)

            # Усредняем время
            run_times = []
            for _ in range(n_runs):
                t0 = time.perf_counter()
                outputs = model.generate(**inputs, max_new_tokens=32,
                                        num_beams=1, do_sample=False)
                run_times.append(time.perf_counter() - t0)

            times.append(min(run_times))  # берём лучшее время
            pred = tokenizer.decode(outputs[0], skip_special_tokens=True).lower()

            # Проверяем по любому ключевому слову
            if any(kw.lower() in pred for kw in item['keywords']):
                correct += 1

    return {
        "accuracy": correct / total,
        "correct": correct,
        "total": total,
        "avg_time_ms": np.mean(times) * 1000,
        "min_time_ms": np.min(times) * 1000,
    }

In [21]:
print("\n--- Computing MI matrices and neuron importance ---")
mi_matrices = {}
neuron_importance = {}

for layer_idx, acts in ffn_activations.items():
    mi_matrices[layer_idx] = compute_mi_matrix(acts)
    neuron_importance[layer_idx] = importance_score(acts)
    avg_mi = mi_matrices[layer_idx].mean().item()
    print(f"  Layer {layer_idx}: avg MI = {avg_mi:.4f}, "
          f"importance range [{neuron_importance[layer_idx].min():.4f}, "
          f"{neuron_importance[layer_idx].max():.4f}]")


--- Computing MI matrices and neuron importance ---
  Layer 0: avg MI = 0.0202, importance range [0.0975, 4.3217]
  Layer 1: avg MI = 0.0188, importance range [0.2196, 7.5561]
  Layer 2: avg MI = 0.0154, importance range [0.1714, 2.8735]
  Layer 3: avg MI = 0.0161, importance range [0.1743, 12.4620]
  Layer 4: avg MI = 0.0155, importance range [0.1533, 2.1939]
  Layer 5: avg MI = 0.0141, importance range [0.1846, 2.4048]
  Layer 6: avg MI = 0.0162, importance range [0.1517, 3.8756]
  Layer 7: avg MI = 0.0150, importance range [0.1037, 7.7004]
  Layer 8: avg MI = 0.0151, importance range [0.1030, 6.7331]
  Layer 9: avg MI = 0.0155, importance range [0.1104, 31.2057]
  Layer 10: avg MI = 0.0173, importance range [0.0693, 2.9664]
  Layer 11: avg MI = 0.0175, importance range [0.0797, 2.4802]
  Layer 12: avg MI = 0.0180, importance range [0.0972, 2.0407]
  Layer 13: avg MI = 0.0174, importance range [0.0508, 1.6043]
  Layer 14: avg MI = 0.0180, importance range [0.0894, 1.7386]
  Layer 15

In [22]:
print("\n" + "="*65)
print("BASELINE (0% pruning)")
print("="*65)
baseline = evaluate_model(base_model, TEST_QA)
baseline_params = count_params(base_model)
print(f"Accuracy:   {baseline['accuracy']*100:.1f}% ({baseline['correct']}/{baseline['total']})")
print(f"Time:       {baseline['avg_time_ms']:.1f} ms avg / {baseline['min_time_ms']:.1f} ms best")
print(f"Parameters: {baseline_params:,}")


BASELINE (0% pruning)
Accuracy:   90.0% (9/10)
Time:       300.6 ms avg / 183.4 ms best
Parameters: 756,155,392


In [25]:
PRUNE_RATIOS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

results = [{
    "prune_ratio": 0.0,
    "pruned_pct": "0%",
    "accuracy": round(baseline['accuracy'], 4),
    "accuracy_pct": round(baseline['accuracy'] * 100, 1),
    "accuracy_drop_pct": 0.0,
    "avg_time_ms": round(baseline['avg_time_ms'], 1),
    "speedup": 1.0,
    "params": baseline_params,
    "params_reduction_pct": 0.0,
    "correct": baseline['correct'],
    "total": baseline['total'],
}]

for ratio in PRUNE_RATIOS:
    print(f"\n{'='*65}")
    print(f"PRUNING {int(ratio*100)}%")
    print("="*65)

    # Определяем нейроны для удаления
    pruned_per_layer = {}
    for layer_idx in mi_matrices:
        pruned = greedy_mi_pruning(
            mi_matrices[layer_idx],
            neuron_importance[layer_idx],
            ratio
        )
        pruned_per_layer[layer_idx] = pruned
        print(f"  Layer {layer_idx}: pruning {len(pruned)}/{D_FF} neurons")

    # Структурированный pruning
    pruned_model = build_pruned_model(base_model, pruned_per_layer)
    pruned_model.eval().to(DEVICE)
    pruned_params = count_params(pruned_model)
    params_reduction = (1 - pruned_params / baseline_params) * 100

    print(f"  Parameters: {pruned_params:,} (↓{params_reduction:.1f}%)")

    # Оценка
    res = evaluate_model(pruned_model, TEST_QA)
    acc_drop = (baseline['accuracy'] - res['accuracy']) * 100
    speedup = baseline['avg_time_ms'] / max(res['avg_time_ms'], 0.1)

    print(f"  Accuracy:   {res['accuracy']*100:.1f}% [{acc_drop:+.1f}%]")
    print(f"  Time:       {res['avg_time_ms']:.1f} ms  [speedup: {speedup:.2f}x]")

    results.append({
        "prune_ratio": ratio,
        "pruned_pct": f"{int(ratio*100)}%",
        "accuracy": round(res['accuracy'], 4),
        "accuracy_pct": round(res['accuracy'] * 100, 1),
        "accuracy_drop_pct": round(acc_drop, 2),
        "avg_time_ms": round(res['avg_time_ms'], 1),
        "speedup": round(speedup, 3),
        "params": pruned_params,
        "params_reduction_pct": round(params_reduction, 2),
        "correct": res['correct'],
        "total": res['total'],
    })

    del pruned_model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


PRUNING 10%
  Layer 0: pruning 281/2816 neurons
  Layer 1: pruning 281/2816 neurons
  Layer 2: pruning 281/2816 neurons
  Layer 3: pruning 281/2816 neurons
  Layer 4: pruning 281/2816 neurons
  Layer 5: pruning 281/2816 neurons
  Layer 6: pruning 281/2816 neurons
  Layer 7: pruning 281/2816 neurons
  Layer 8: pruning 281/2816 neurons
  Layer 9: pruning 281/2816 neurons
  Layer 10: pruning 281/2816 neurons
  Layer 11: pruning 281/2816 neurons
  Layer 12: pruning 281/2816 neurons
  Layer 13: pruning 281/2816 neurons
  Layer 14: pruning 281/2816 neurons
  Layer 15: pruning 281/2816 neurons
  Layer 16: pruning 281/2816 neurons
  Layer 17: pruning 281/2816 neurons
  Layer 18: pruning 281/2816 neurons
  Layer 19: pruning 281/2816 neurons
  Layer 20: pruning 281/2816 neurons
  Layer 21: pruning 281/2816 neurons
  Layer 22: pruning 281/2816 neurons
  Layer 23: pruning 281/2816 neurons
  Parameters: 735,437,824 (↓2.7%)
  Accuracy:   100.0% [-10.0%]
  Time:       321.5 ms  [speedup: 0.94x]

PRU

In [27]:
print("\n\n" + "="*80)
print("FINAL RESULTS SUMMARY — MI Pruning on Kyrmasch/t5-kazakh-qa")
print("="*80)
header = f"{'Pruning':>8} | {'Accuracy':>10} | {'Acc Drop':>9} | {'Params ↓':>8} | {'Time ms':>9} | {'Speedup':>8}"
print(header)
print("-" * 80)
for r in results:
    print(f"{r['pruned_pct']:>8} | "
          f"{r['accuracy_pct']:>9.1f}% | "
          f"{r['accuracy_drop_pct']:>8.1f}% | "
          f"{r['params_reduction_pct']:>7.1f}% | "
          f"{r['avg_time_ms']:>9.1f} | "
          f"{r['speedup']:>7.2f}x")

# Сохранение
out_path = "outputs/mi_pruning_v2_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump({
        "model": MODEL_NAME,
        "baseline_params": baseline_params,
        "d_ff": D_FF,
        "n_encoder_layers": len(base_model.encoder.block),
        "results": results,
    }, f, ensure_ascii=False, indent=2)

print(f"\nResults saved to {out_path}")



FINAL RESULTS SUMMARY — MI Pruning on Kyrmasch/t5-kazakh-qa
 Pruning |   Accuracy |  Acc Drop | Params ↓ |   Time ms |  Speedup
--------------------------------------------------------------------------------
      0% |      90.0% |      0.0% |     0.0% |     300.6 |    1.00x
     10% |     100.0% |    -10.0% |     2.7% |     321.5 |    0.94x
     20% |     100.0% |    -10.0% |     5.5% |     333.0 |    0.90x
     30% |      90.0% |      0.0% |     8.2% |     337.2 |    0.89x
     40% |      90.0% |      0.0% |    11.0% |     336.0 |    0.90x
     50% |      70.0% |     20.0% |    13.7% |     383.6 |    0.78x
     60% |      40.0% |     50.0% |    16.5% |     523.9 |    0.57x
     70% |      20.0% |     70.0% |    19.2% |     886.8 |    0.34x

Results saved to outputs/mi_pruning_v2_results.json
